# Reasoning without Observation (Suy luận không cần quan sát)

Đây là một mô hình agent kết hợp bộ lập kế hoạch đa bước (multi-step planner) và thay thế biến để sử dụng công cụ một cách hiệu quả. Phương pháp này được thiết kế để cải thiện kiến trúc agent dạng ReACT theo các hướng sau:

1. Giảm tiêu thị token và thời gian thực thi: Bằng cách tạo ra toàn bộ chuỗi công cụ sẽ sử dụng chỉ trong 1 lần duy nhất (_Kiến trúc ReACT yêu cầu nhiều lần gọi LLM với các phần tiền tố dư thừa, vì prompt hệ thống và các bước trước đó phải được gửi lại cho LLM ở mỗi bước suy luận_)
2. Đơn giản hoá quá trình fine tuning: Vì dữ liệu lập kế hoạch không phụ thuộc vào kết quả đầu ra của công cụ, các mô hình có thể được tinh chỉnh mà không cần thực sự gọi các công cụ đó (về mặt lý thuyết)

Sơ đồ dưới đây phác thảo biểu đồ tính toán tổng thể của ReWOO:

![ReWoo Diagram](diagram.png)

ReWOO được cấu thành từ 3 mô-đun:

1. 🧠**Planner**: Tạo kế hoạch theo định dạng sau:
```text
Plan: <reasoning>
#E1 = Tool[argument for tool]
Plan: <reasoning>
#E2 = Tool[argument for tool with #E1 variable substitution]
...
```
3. **Worker**: Thực hiện công cụ với các đối số đã được cung cấp.
4. 🧠**Solver**: Tạo câu trả lời cho nhiệm vụ ban đầu dựa trên các quan sát thu được từ công cụ.

Các mô-đun có biểu tượng 🧠 phụ thuộc vào việc gọi LLM. Lưu ý rằng chúng ta tránh được các lần gọi lặp lại tới Planner LLM bằng cách sử dụng cơ chế thay thế biến.

In [2]:
%%capture --no-stderr
%pip install -U langgraph langchain_community langchain_openai tavily-python

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

os.environ["TAVILY_API_KEY"] = "YOUR_TAVILY_API_KEY"

model = ChatOpenAI(model="gpt-4o", temperature=0)
search = TavilySearchResults()

In [4]:
from typing import TypedDict, List, Dict, Any

class ReWOO(TypedDict):
    task: str
    plan_string: str # output cúa planner 
    steps: List[Any] # Danh sách các bước đã parse từ plan
    results: Dict[str, str] # Dictionary lưu kết quả của từng bước
    result: str # Kết quả cuối cùng sau khi thực hiện tất cả các bước

## Planner
Planner sẽ prompt một LLM để tạo ra một kế hoạch dưới dạng danh sách các nhiệm vụ. Các tham số của mỗi nhiệm vụ là các chuỗi ký tự, và có thể chứa các biến đặc biệt dạng (`#E{0-9}+`). Những biến này được dùng để thay thế giá trị từ kết quả của các nhiệm vụ trước đó.

![workflow](workflow.png)


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="deepseek/deepseek-r1",
    base_url="https://openrouter.ai/api/v1",
    api_key="YOUR_OPENROUTER_API_KEY",
)

In [11]:
prompt = """Đối với nhiệm vụ sau, hãy lập kế hoạch để giải quyết vấn đề từng bước.
Đối với mỗi kế hoạch (plan), hãy chỉ rõ công cụ bên ngoài cần sử dụng cùng với input của công cụ để thu thập thông tin.
Bạn có thể lưu thông tin thu được vào một biến #E để có thể sử dụng lại ở các bước sau.
(Plan, #E1, Plan, #E2, Plan, ...)

Các công cụ có thể sử dụng:

(1) Google[input]: Worker dùng để tìm kiếm kết quả từ Google.
Hữu ích khi bạn cần tìm câu trả lời ngắn gọn và chính xác về một chủ đề cụ thể.
Input phải là một truy vấn tìm kiếm.

(2) LLM[input]: Một mô hình ngôn ngữ đã được huấn luyện trước giống như bạn.
Hữu ích khi bạn cần sử dụng kiến thức chung và suy luận thông thường.
Ưu tiên sử dụng khi bạn tự tin có thể giải quyết vấn đề bằng suy luận.
Input có thể là bất kỳ hướng dẫn nào.

Ví dụ:

Task: Thomas, Toby và Rebecca làm tổng cộng 157 giờ trong một tuần.
Thomas làm x giờ.
Toby làm ít hơn 10 giờ so với gấp đôi số giờ Thomas làm,
và Rebecca làm ít hơn Toby 8 giờ.
Rebecca đã làm bao nhiêu giờ?

Plan: Vì Thomas làm x giờ, hãy chuyển bài toán thành biểu thức đại số và giải bằng Wolfram Alpha.
#E1 = WolframAlpha[Solve x + (2x − 10) + ((2x − 10) − 8) = 157]

Plan: Tìm số giờ Thomas đã làm.
#E2 = LLM[What is x, given #E1]

Plan: Tính số giờ Rebecca đã làm.
#E3 = Calculator[(2 * #E2 − 10) − 8]

Begin!

Hãy mô tả kế hoạch của bạn với đầy đủ chi tiết.
Mỗi Plan phải đi kèm đúng một biến #E.

Task: {task}"""

In [12]:
task = "Quê hương chính xác của nhà vô địch Úc mở rộng năm 2024 nam là gì?"

In [19]:
result = model.invoke(prompt.format(task=task))

In [20]:
print(result.content)

Plan: Xác định nhà vô địch Úc mở rộng 2024 nam đơn.  
#E1 = Google[Ai là nhà vô địch Úc mở rộng 2024 nam đơn?]  

Plan: Dùng thông tin từ #E1 để tìm quê hương của vận động viên.  
#E2 = Google[Quê hương của [tên vận động viên từ #E1] là đâu?]  

Plan: Xác nhận thông tin quê hương thông qua dữ liệu đáng tin cậy.  
#E3 = LLM[Kiểm tra lại thông tin "#E2" có chính xác không dựa trên kiến thức nền về quê hương của vận động viên này?]


#### Planner Node

Để kết nối planner vào graph, chúng ta sẽ tạo một node `get_plan`. Node này tiếp nhận trạng thái ReWOO và trả về bản cập nhật trạng thái cho các trường `steps` và `plan_string`.

In [5]:
import re

text = """
Plan: Tìm ca sĩ chính của Foo Fighters.
#E1 = Tavily[lead singer of Foo Fighters]

Plan: Tìm tuổi của ca sĩ chính.
#E2 = Tavily[age of #E1]
"""

regex_pattern = r"Plan:\s*(.*?)\s*\n\s*(#E\d+)\s*=\s*([A-Za-z_]\w*)\s*\[(.*?)\]"

matches = re.findall(regex_pattern, text, re.DOTALL)

for m in matches:
    print(m)

('Tìm ca sĩ chính của Foo Fighters.', '#E1', 'Tavily', 'lead singer of Foo Fighters')
('Tìm tuổi của ca sĩ chính.', '#E2', 'Tavily', 'age of #E1')


In [22]:
import re 
from langchain_core.prompts import ChatPromptTemplate

regex_pattern = r"Plan:\s*(.*?)\s*\n\s*(#E\d+)\s*=\s*([A-Za-z_]\w*)\s*\[(.*?)\]"

planner_prompt = ChatPromptTemplate.from_messages([("system", prompt)]) # Tạo một prompt template từ chuỗi prompt đã định nghĩa
planner = planner_prompt | model # Kết hợp prompt template với mô hình ngôn ngữ để tạo ra một planner có khả năng lập kế hoạch dựa trên prompt đã định nghĩa

def get_plan(state: ReWOO):
    task = state["task"]
    result = planner.invoke({"task": task}) # Gọi planner với task cụ thể để nhận được kế hoạch giải quyết nhiệm vụ
    matches = re.findall(regex_pattern, result.content)
    return {
        "steps": matches,
        "plan_string": result.content
    }

print(get_plan({"task": task}))

{'steps': [('Tìm tên của nhà vô địch nam Úc Mở rộng 2024.', '#E1', 'Google', "Australian Open 2024 men's singles champion"), ('Xác định quốc gia quê hương của nhà vô địch dựa trên thông tin từ #E1.', '#E2', 'Google', 'What is the nationality of Jannik Sinner?'), ('Kết luận quê hương chính xác dựa trên kết quả #E2.', '#E3', 'LLM', "Summarize #E2 to state Jannik Sinner's home country.")], 'plan_string': 'Hmm, mình cần tìm quê hương chính xác của nhà vô địch Úc mở rộng năm 2024 nam. Đầu tiên, mình nhớ rằng giải Úc mở rộng 2024 đã diễn ra vào tháng 1 năm 2024. Nhà vô địch nam đơn có lẽ là Jannik Sinner, nhưng mình không chắc lắm. Để xác nhận, mình nên tìm thông tin từ nguồn đáng tin cậy.\n\nBước đầu tiên, mình sẽ dùng công cụ để tìm kiếm thông tin về nhà vô địch nam Úc mở rộng 2024. Mình sẽ dùng Google với từ khóa "Australian Open 2024 men\'s singles champion". Kết quả trả về sẽ cho biết tên của vận động viên đó. Sau khi xác định được tên, mình cần biết quê hương của họ. Ví dụ, nếu là Jann

## Executor 
Bộ thực thi nhận kế hoạch và tiến hành chạy các tools theo trình tự đã đề ra

In [23]:
from langchain_community.tools.tavily_search import TavilySearchResults

search = TavilySearchResults()

In [36]:
def _get_current_task(state: ReWOO):
    if state.get("results") is None: 
        return 1 # chưa có kết quả nào thì chạy step 1
    if len(state["results"]) == len(state["steps"]): # chạy đủ step
        return None
    else:
        return len(state['results']) + 1 # chạy step tiếp theo

In [37]:
def _get_current_task(state: ReWOO):
    if "results" not in state or state["results"] is None:
        return 1
    if len(state["results"]) == len(state["steps"]):
        return None
    else:
        return len(state["results"]) + 1


def tool_execution(state: ReWOO):
    _step = _get_current_task(state)
    _, step_name, tool, tool_input = state["steps"][_step - 1]
    _results = (state["results"] or {}) if "results" in state else {}
    for k, v in _results.items():
        tool_input = tool_input.replace(k, v)
    if tool == "Google":
        result = search.invoke(tool_input)
    elif tool == "LLM":
        result = model.invoke(tool_input)
    else:
        raise ValueError
    _results[step_name] = str(result)
    return {"results": _results}

## Solver

Solver nhận toàn bộ kế hoạch và tạo ra phản hồi cuối cùng dựa trên kết quả trả về từ lần gọi công cụ của worker 

In [29]:
solve_prompt = """Hãy giải quyết nhiệm vụ hoặc vấn đề sau đây. Để giải quyết vấn đề, chúng tôi đã lập một Kế hoạch (Plan) từng bước và truy xuất các Bằng chứng (Evidence) tương ứng cho mỗi bước. Hãy sử dụng chúng một cách cẩn trọng vì các bằng chứng dài có thể chứa thông tin không liên quan.

{plan}

Bây giờ, hãy giải quyết câu hỏi hoặc nhiệm vụ dựa trên các Bằng chứng đã được cung cấp ở trên. Trả lời trực tiếp kết quả, không thêm từ ngữ dư thừa.

Task: {task}
Response:"""

def solve(state: ReWOO):
    plan = ""
    for _plan, step_name, tool, tool_input in state["steps"]:
        _results = (state["results"] or {}) if "results" in state else {}
        for k, v in _results.items():
            tool_input = tool_input.replace(k, v)
            step_name = step_name.replace(k, v)
        plan += f"Plan: {_plan}\n{step_name} = {tool}[{tool_input}]"
        # Plan: Tìm nhà vô địch Úc mở rộng 2024
        # Jannik Sinner = Google[2024 men's Australian Open winner]
    prompt = solve_prompt.format(plan=plan, task=state["task"])
    result = model.invoke(prompt)
    return {"result": result.content}

## Define Graph
Graph sẽ định nghĩa toàn bộ workflow. Mỗi module bao gồm: planner, tool executor và solver đều được thêm vào dưới dạng node trong đồ thị

In [31]:
def _route(state):
    _step = _get_current_task(state)
    if _step is None:
        # hoàn thành tất cả các bước, chuyển sang node "solve" để giải quyết nhiệm vụ
        return "solve"
    else:
        return "tool"

In [38]:
from langgraph.graph import END, StateGraph, START

graph = StateGraph(ReWOO)
graph.add_node("plan", get_plan)
graph.add_node("tool", tool_execution)
graph.add_node("solve", solve)
graph.add_edge("plan", "tool")
graph.add_edge("solve", END)
# Sau khi thực hiện một bước công cụ, kiểm tra xem còn bước nào chưa thực hiện hay không. 
# Nếu còn, quay lại node "tool" để thực hiện bước tiếp theo. 
# Nếu đã hoàn thành tất cả các bước, chuyển sang node "solve" để giải quyết nhiệm vụ.
graph.add_conditional_edges("tool", _route) 
graph.add_edge(START, "plan")

app = graph.compile()

In [39]:
for s in app.stream({"task": task}):
    print(s)
    print("---")

{'plan': {'steps': [('Tìm tên của nhà vô địch nam Úc Mở rộng 2024.', '#E1', 'Google', "Australian Open 2024 men's singles champion"), ('Xác định quốc tịch/quê hương của vận động viên dựa trên kết quả #E1.', '#E2', 'LLM', 'What is the nationality of Jannik Sinner?'), ('Kiểm tra chéo thông tin quê hương để đảm bảo độ chính xác.', '#E3', 'Google', 'Jannik Sinner nationality')], 'plan_string': 'Ồ, mình cần tìm quê hương chính xác của nhà vô địch Úc mở rộng năm 2024 nam. Đầu tiên, mình nhớ giải Úc mở rộng 2024 là một giải Grand Slam tennis. Nhưng không chắc ai đã thắng giải nam. Mình cần xác định tên của nhà vô địch nam năm đó. Sau khi có tên, mình sẽ tìm thông tin về quê hương của người đó. \n\nCó thể mình sẽ dùng Google để tìm thông tin về nhà vô địch nam Úc mở rộng 2024. Tìm kiếm với từ khóa "Australian Open 2024 men\'s singles champion". Kết quả sẽ cho biết tên người thắng. Sau đó, mình tiếp tục tìm kiếm thông tin về quốc tịch của người đó. \n\nNếu Google không đủ, mình có thể dùng LLM 

In [41]:
print(s["solve"]["result"])

Innichen (San Candido), South Tyrol, Italy.
